# CRNN Combined Training (Kaggle + IAM)

Run this notebook on Google Colab with a T4 GPU runtime.

1. Runtime → Change runtime type → T4 GPU
2. Run all cells
3. Checkpoint will be saved to Google Drive

In [ ]:
# Clone repo and install deps
!git clone https://github.com/inSideos-designs/handwriting-ocr.git
%cd handwriting-ocr
!pip install -q torch torchvision numpy Pillow pandas opencv-python-headless scikit-learn datasets google-cloud-storage

In [ ]:
# Check GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Download Kaggle dataset from GCS
!pip install -q gcsfs
!gsutil cp gs://doc-processing-hw-ocr-ml/data/dataset.tar.gz /tmp/dataset.tar.gz

import tarfile, os
os.makedirs('/tmp/kaggle', exist_ok=True)
with tarfile.open('/tmp/dataset.tar.gz', 'r:gz') as tar:
    tar.extractall(path='/tmp/kaggle')
os.remove('/tmp/dataset.tar.gz')
print('Kaggle dataset extracted')

In [ ]:
# If GCS auth fails, download from Kaggle directly instead
# Uncomment these lines and upload your kaggle.json:
# !pip install -q kaggle
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d landlord/handwriting-recognition -p /tmp/kaggle --unzip

In [ ]:
# Prepare Kaggle data
from orchestration.assets.data_prep import load_and_clean_labels, validate_images, split_dataset

train_csv = '/tmp/kaggle/written_name_train_v2.csv'
img_dir = '/tmp/kaggle/train_v2/train'

df = load_and_clean_labels(train_csv)
df = validate_images(df, img_dir)
train_df, val_df = split_dataset(df, val_ratio=0.1)

os.makedirs('/tmp/processed', exist_ok=True)
train_df.to_csv('/tmp/processed/train.csv', index=False)
val_df.to_csv('/tmp/processed/val.csv', index=False)
print(f'Kaggle: {len(train_df)} train, {len(val_df)} val')

In [ ]:
# Load datasets
from model.data.dataset import HandwritingDataset, NUM_CLASSES
from model.data.combined_dataset import IAMWordDataset
from torch.utils.data import ConcatDataset, DataLoader

kaggle_train = HandwritingDataset('/tmp/processed/train.csv', img_dir)
kaggle_val = HandwritingDataset('/tmp/processed/val.csv', img_dir)

iam_train = IAMWordDataset(split='train', max_label_len=32, target_width=128)
iam_val = IAMWordDataset(split='validation', max_label_len=32, target_width=128)

train_ds = ConcatDataset([kaggle_train, iam_train])
val_ds = ConcatDataset([kaggle_val, iam_val])
print(f'Combined: {len(train_ds)} train, {len(val_ds)} val')

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=4)

In [ ]:
# Train
import torch.nn as nn
from model.networks.crnn import CRNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

model = CRNN(32, 1, NUM_CLASSES, 256, 2, 0.1).to(device)
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=3)

best_val = float('inf')
os.makedirs('checkpoints', exist_ok=True)

for epoch in range(15):
    model.train()
    total = 0.0
    for batch_idx, (imgs, targets, tlens) in enumerate(train_loader):
        imgs = imgs.to(device)
        out = model(imgs)
        sl, bs = out.shape[0], out.shape[1]
        il = torch.full((bs,), sl, dtype=torch.long)
        tf = []
        for i in range(bs):
            tf.extend(targets[i, :tlens[i]].tolist())
        tf = torch.tensor(tf, dtype=torch.long)
        loss = criterion(out, tf, il, tlens)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total += loss.item()
        if batch_idx % 500 == 0:
            print(f'  Epoch {epoch+1}, batch {batch_idx}/{len(train_loader)}, loss={loss.item():.4f}')
    tl = total / len(train_loader)

    model.eval()
    vt = 0.0
    with torch.no_grad():
        for imgs, targets, tlens in val_loader:
            imgs = imgs.to(device)
            out = model(imgs)
            sl, bs = out.shape[0], out.shape[1]
            il = torch.full((bs,), sl, dtype=torch.long)
            tf = []
            for i in range(bs):
                tf.extend(targets[i, :tlens[i]].tolist())
            tf = torch.tensor(tf, dtype=torch.long)
            loss = criterion(out, tf, il, tlens)
            vt += loss.item()
    vl = vt / len(val_loader)
    scheduler.step(vl)
    print(f'Epoch {epoch+1}/15: train={tl:.4f}, val={vl:.4f}')

    if vl < best_val:
        best_val = vl
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(), 'val_loss': vl},
                   'checkpoints/best_model.pt')
        print(f'  Saved checkpoint (val={vl:.4f})')

print('Training complete!')

In [ ]:
# Upload checkpoint to GCS
!gsutil cp checkpoints/best_model.pt gs://doc-processing-hw-ocr-ml/checkpoints/best_model.pt
print('Checkpoint uploaded to GCS')

In [ ]:
# Also save to Google Drive as backup
from google.colab import drive
drive.mount('/content/drive')
!cp checkpoints/best_model.pt /content/drive/MyDrive/best_model.pt
print('Saved to Google Drive')

In [ ]:
# Quick test on Kaggle samples
from model.inference.predictor import Predictor
from model.training.config import TrainConfig
from PIL import Image

config = TrainConfig()
predictor = Predictor('checkpoints/best_model.pt', config)

for i in ['00001', '00002', '00003', '00010']:
    img = Image.open(f'/tmp/kaggle/train_v2/train/TRAIN_{i}.jpg')
    text, conf = predictor.predict_with_confidence(img)
    print(f'TRAIN_{i}: "{text}" (conf={conf:.3f})')

In [ ]:
# Test on IAM samples
from datasets import load_dataset
ds = load_dataset('Teklia/IAM-line', split='test')

for i in range(5):
    text, conf = predictor.predict_with_confidence(ds[i]['image'])
    print(f'Truth: "{ds[i]["text"]}"')
    print(f'Pred:  "{text}" (conf={conf:.3f})')
    print()